# Introduction to Verification & Validation for AI Systems

**Audience:** Students with ML/programming background but no V&V experience.

**What you'll learn:**
1. What V&V means and why it matters for AI
2. How to write *good* requirements (the INCOSE way)
3. How to build a specification, create evidence, and generate reports
4. How to trace everything back to safety standards

**Time:** ~30 minutes

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

# If running from the repo root, use: sys.path.insert(0, ".")
# Install: pip install vnvspec matplotlib

import vnvspec
print(f"vnvspec version: {vnvspec.__version__}")

In [ ]:
# Import our visualization helpers (same directory as this notebook)
from _helpers import (
    display_requirements_table,
    display_violations_table,
    display_evidence_table,
    display_report_summary,
    display_requirement_card,
    plot_violations_by_rule,
    plot_evidence_verdicts,
    plot_coverage,
    display_registry_sample,
)
print("Helpers loaded.")

## 2. What is Verification & Validation?

Think of building a self-driving car's perception system:

| Question | V&V Term | Meaning |
|----------|----------|---------|
| "Are we building it right?" | **Verification** | Does the system meet its *specifications*? |
| "Are we building the right thing?" | **Validation** | Does the system meet the *user's actual needs*? |

**Without V&V**, you might train a model that scores 99% accuracy on a benchmark
but fails catastrophically in fog, at night, or on unusual road signs.

**With V&V**, you write down *exactly* what the system must do (requirements),
*prove* it does those things (evidence), and *trace* everything to safety standards.

Let's see how `vnvspec` makes this concrete.

## 3. Your First Requirement

A **requirement** is a single, testable statement about what the system *shall* do.

Let's write one:

In [ ]:
from vnvspec import Requirement

# A well-written requirement
req_good = Requirement(
    id="REQ-001",
    statement="The classifier shall produce output probabilities in the range [0.0, 1.0].",
    rationale="Downstream calibration requires valid probability outputs.",
    verification_method="test",
    acceptance_criteria=[
        "All output values are between 0.0 and 1.0 inclusive.",
        "Output probabilities sum to 1.0 within tolerance of 0.01.",
    ],
    priority="high",
)

display_requirement_card(req_good)

## 4. Good vs. Bad Requirements — The GtWR Checker

The **INCOSE Guide to Writing Requirements (GtWR)** defines 8 rules for requirement quality.
vnvspec has a built-in checker. Let's compare a good and a bad requirement:

In [ ]:
# Check the good requirement
violations = req_good.check_quality()
print(f"Good requirement violations: {len(violations)}")
display_violations_table(violations)

In [ ]:
# Now a BAD requirement — vague, ambiguous, no criteria
req_bad = Requirement(
    id="bad-req",
    statement="The system should probably work fast and safely",
    rationale="",                # no rationale!
    verification_method="test",
    acceptance_criteria=[],      # no criteria!
)

violations_bad = req_bad.check_quality()
print(f"Bad requirement violations: {len(violations_bad)}")
display_violations_table(violations_bad)

In [ ]:
# Visualize the violations
plot_violations_by_rule(violations_bad, title="What's wrong with the bad requirement?")

**Key takeaway:** Good requirements are *singular* (one thing), *unambiguous* (no "should" or "maybe"),
*verifiable* (clear pass/fail criteria), and *necessary* (justified by a rationale).

## 5. Building a Specification

A **Spec** groups requirements together with hazards, contracts, and evidence.
Let's create a spec for a simple image classifier:

In [ ]:
from vnvspec import Spec, Requirement

# Define 4 requirements
requirements = [
    Requirement(
        id="REQ-PROB",
        statement="The classifier shall produce probabilities in [0.0, 1.0].",
        rationale="Valid probability outputs for calibration.",
        verification_method="test",
        acceptance_criteria=["All outputs in [0.0, 1.0]."],
        priority="high",
    ),
    Requirement(
        id="REQ-DIM",
        statement="The classifier shall output exactly 3 class scores.",
        rationale="The dataset has 3 classes.",
        verification_method="test",
        acceptance_criteria=["Output dimension equals 3."],
    ),
    Requirement(
        id="REQ-NAN",
        statement="The classifier shall not produce NaN values.",
        rationale="NaN propagation causes silent failures.",
        verification_method="test",
        acceptance_criteria=["No NaN in any output tensor."],
        priority="high",
    ),
    Requirement(
        id="REQ-LATENCY",
        statement="The classifier shall produce a prediction within 50 ms per sample.",
        rationale="Real-time inference budget.",
        verification_method="test",
        acceptance_criteria=["p99 latency < 50 ms."],
    ),
]

spec = Spec(name="image-classifier-v1", requirements=requirements)
print(f"Spec '{spec.name}' has {len(spec.requirements)} requirements")
display_requirements_table(spec.requirements)

In [ ]:
# Batch quality check
print("Quality check for all requirements:\n")
all_violations = []
for req in spec.requirements:
    v = req.check_quality()
    all_violations.extend(v)
    status = "PASS" if not v else f"{len(v)} issues"
    print(f"  {req.id}: {status}")

print(f"\nTotal violations: {len(all_violations)}")

## 6. A Simple "Model"

You don't need PyTorch to understand V&V! Let's use a plain Python function as our "model":

In [ ]:
import random
import math

def simple_classifier(features: list[float]) -> list[float]:
    """A fake classifier that returns 3 class probabilities."""
    # Simulate some logits
    logits = [sum(f * w for f, w in zip(features, [0.3, -0.1, 0.5])) + random.gauss(0, 0.1)
              for _ in range(3)]
    # Softmax
    max_l = max(logits)
    exps = [math.exp(l - max_l) for l in logits]
    total = sum(exps)
    return [e / total for e in exps]

# Test it
sample_input = [0.5, 1.2, -0.3]
output = simple_classifier(sample_input)
print(f"Input:  {sample_input}")
print(f"Output: {[f'{p:.4f}' for p in output]}")
print(f"Sum:    {sum(output):.6f}")

## 7. Creating Evidence

Now let's *verify* our model against the spec by running it on test data
and recording **Evidence** objects:

In [ ]:
from vnvspec import Evidence
from datetime import datetime, UTC
import math

# Generate test data
test_data = [[random.gauss(0, 1) for _ in range(3)] for _ in range(100)]

# Run the model
outputs = [simple_classifier(x) for x in test_data]

# --- REQ-PROB: probabilities in [0, 1]? ---
all_in_range = all(0.0 <= p <= 1.0 for out in outputs for p in out)
ev_prob = Evidence(
    id="EV-PROB-001",
    requirement_id="REQ-PROB",
    kind="test",
    verdict="pass" if all_in_range else "fail",
    details={"samples": len(test_data), "all_in_range": all_in_range},
)

# --- REQ-DIM: output dimension == 3? ---
all_dim3 = all(len(out) == 3 for out in outputs)
ev_dim = Evidence(
    id="EV-DIM-001",
    requirement_id="REQ-DIM",
    kind="test",
    verdict="pass" if all_dim3 else "fail",
    details={"samples": len(test_data)},
)

# --- REQ-NAN: no NaN? ---
has_nan = any(math.isnan(p) for out in outputs for p in out)
ev_nan = Evidence(
    id="EV-NAN-001",
    requirement_id="REQ-NAN",
    kind="test",
    verdict="fail" if has_nan else "pass",
    details={"nan_count": sum(1 for out in outputs for p in out if math.isnan(p))},
)

# --- REQ-LATENCY: not tested yet ---
ev_latency = Evidence(
    id="EV-LAT-001",
    requirement_id="REQ-LATENCY",
    kind="test",
    verdict="inconclusive",
    details={"note": "Latency benchmark not yet run."},
)

evidence = [ev_prob, ev_dim, ev_nan, ev_latency]
display_evidence_table(evidence)

## 8. Building a Report

In [ ]:
from vnvspec.core.assessment import Report

report = Report(
    spec_name=spec.name,
    spec_version=spec.version,
    evidence=evidence,
    summary={"test_samples": 100, "model": "simple_classifier"},
)

display_report_summary(report)

In [ ]:
# Visualize verdicts
plot_evidence_verdicts(report.evidence, title="How did our model do?")

## 9. Coverage — Are All Requirements Tested?

In [ ]:
# Add evidence to the spec to check coverage
spec_with_ev = Spec(
    name=spec.name,
    requirements=spec.requirements,
    evidence=evidence,
)

print("Coverage summary:", spec_with_ev.coverage_summary())
plot_coverage(spec_with_ev)

## 10. Exporting Your Report

vnvspec can export reports in multiple formats:

In [ ]:
from vnvspec.exporters import export_markdown, export_gsn_mermaid

# Markdown report
md_report = export_markdown(report)
print(md_report)

In [ ]:
# GSN assurance case (Mermaid diagram)
gsn = export_gsn_mermaid(report)
print(gsn)

In [ ]:
from _helpers import display_mermaid
display_mermaid(gsn)

## 11. Traceability — Connecting the Dots

Real V&V requires *tracing* requirements to hazards, evidence, and standards.

In [ ]:
from vnvspec import TraceLink, Hazard, build_trace_graph
from vnvspec.core.trace import coverage_report
from _helpers import plot_trace_graph

# Define a hazard
haz = Hazard(
    id="HAZ-001",
    description="Incorrect classification leads to wrong action.",
    severity="S3", exposure="E4", controllability="C3", asil="D",
    mitigations=["REQ-PROB", "REQ-NAN"],
)

# Create trace links
links = [
    TraceLink(source_id="REQ-PROB", target_id="HAZ-001", relation="mitigates"),
    TraceLink(source_id="REQ-NAN", target_id="HAZ-001", relation="mitigates"),
    TraceLink(source_id="EV-PROB-001", target_id="REQ-PROB", relation="verifies"),
    TraceLink(source_id="EV-NAN-001", target_id="REQ-NAN", relation="verifies"),
    TraceLink(source_id="EV-DIM-001", target_id="REQ-DIM", relation="verifies"),
    TraceLink(source_id="REQ-PROB", target_id="ISO-8800-6.2", relation="maps_to_standard"),
]

plot_trace_graph(links)

## 12. Standards Registries

vnvspec ships with clause databases for major safety standards:

In [ ]:
from vnvspec.registries import list_available
print("Available registries:", list_available())

In [ ]:
display_registry_sample("iso_pas_8800", n=8)

## Summary

| Concept | What it is | vnvspec class |
|---------|-----------|---------------|
| Requirement | A testable "shall" statement | `Requirement` |
| Specification | Collection of requirements + context | `Spec` |
| Evidence | Proof that a requirement is met | `Evidence` |
| Hazard | An identified risk | `Hazard` |
| Traceability | Links between all artifacts | `TraceLink` |
| Report | Assessment results | `Report` |

**Next steps:**
- Try Notebook 2 (SE perspective) for standards compliance and formal traceability
- Try Notebook 3 (ML perspective) for PyTorch model wrapping and automated assessment